In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# Dynamic programming: solving the Bellman equation with a model

## Where this session fits

- **Series goal**
    - choose good actions by trial and error
    - from small tables to robots and language models
- **Position: Part A (value-based methods), session 2**
    - the last session defined the value and the Bellman equation, in an average form and a max form
    - **the limitation we start from:** the equation has the value on both sides, so it is not yet an answer
    - this session solves it exactly, by iterating, on a problem whose rules (the model) are known
- **The question:** can we solve the Bellman equation exactly?
- **By the end you can**
    - run value iteration and policy iteration on a small grid
    - say how policy iteration moves the max out of the sweep
    - say what *evaluate* and *improve* are, and how both algorithms are loops of them
    - say what dynamic programming needs, and why that is its limitation


# Model-based Value methods

**Why this section?** The Bellman equation gives a recipe for computing values when the model is known. Two algorithms use it: value iteration and policy iteration.


We will study two common Model-based, Dynamic Programming type methods
- Value Iteration
- Policy Iteration

and their associated Bellman equations.

Terms used below: a **sweep** is one pass of the update over every state (the index $k$ counts sweeps). A **round** of policy iteration is one evaluation followed by one improvement. A policy is **greedy** with respect to a value if it picks the action with the best one-step lookahead. The lake is the one of the last session: squares $0$ to $15$ row by row, non-slippery, discount $0.9$.



## Value iteration method

The simplest method is to
- iteratively update the Value function
    - until convergence
- derive the *final* Policy from the Value function
    - choose the action with the best one-step lookahead value
    - based on the Value function

The estimate of the value of a state $\state$ is improved iteratively.

Let 
$$
\statevalfun_{k}(\state) 
$$

denote the *estimated* value of state $\state$ after $k$ sweeps.

In sweep $k+1$, the value of *each* state $\state \in \States$ is updated via the Bellman update


$$
\statevalfun_{k+1}(\state) = \max{\act} \sum_{\state', \rew} \transp(\state', \rew \mid \state, \act)\bigl(\rew + \disc \statevalfun_k(\state')\bigr).
$$

where $\rew$ is the reward the environment returns together with the successor state $\state'$ (the model gives the chance of each pair $(\state', \rew)$)

- the fixed point of this update is the Bellman *optimality* equation
$$
\statevalfun_*(\state) = \max{\act} \sum_{\state', \rew} \transp(\state', \rew \mid \state, \act)\bigl(\rew + \disc \statevalfun_*(\state')\bigr)
$$
- value iteration repeats the update until the table stops changing


That is
- $\statevalfun_{k+1}(\state)$ 
- uses the *previous sweep's $(k)$* estimate $$\statevalfun_k(\state')$$  
of each successor state $\state'$
- and chooses the action leading to highest return

The backup in each sweep
- moves information about potential future (successor) states
- to the current state

So the Bellman update **propagates information (expected return) from deeper states to shallower ones**
- *deeper*: later in the episode, closer to where the reward is earned (here: near **G**)
- *shallower*: earlier in the episode (here: near **S**)
- the value-iteration pictures a few slides ahead show exactly this


For Frozen Lake
- the only non-zero reward is obtained at terminal state (the Gift) G

Thus, in sweep $1$
- the only non-zero information is the reward for entering G, which reaches its predecessor, square $14$

This information can only diffuse to more distant predecessors 
- through multiple sweeps

The previous sweep's estimate of the successors of $\state$
$$\statevalfun_k(\state')$$  

in turn, are based on the sweep $(k-1)$ values of the successors of $\state'$

So we need $i$ sweeps
- in order for information of states $i$ steps ahead
- to reach state $\state$

Hence the need to repeat until convergence.


**Value iteration on Frozen Lake** (non-slippery, discount $0.9$)

<img src="images/value_iteration_strip.png" width=100%>

- sweep 1: only the square next to the goal gains value
- each further sweep moves information one square back from G: from deeper states to shallower ones
- after $6$ sweeps the values stop changing (sweep $7$ confirms that nothing changes): they are the optimal values, and the arrows are the optimal policy
- every square is updated from the *previous* sweep's table (synchronous)
- worked backup, square $10$ at sweep $2$: down reaches square $14$, worth $1.00$ after sweep $1$, so $\statevalfun_2(10) = 0 + 0.9 \times 1.00 = 0.90$ (right is a hole)
- sweeps $4$ and $5$ (not shown) give squares $2$ and $8$ the value $0.73$, then squares $1$, $3$ and $4$ the value $0.66$
- the pseudo-code on the next slides overwrites $\statevalfun$ in place instead
- the result is the same


**Subtlety**

Prior to convergence
- $\statevalfun_k(\state)$ is a *non-random* variable
    - $\transp(\state', \rew \mid \state, \act)$ is *known* for all $\state, \act$ since method is model-based
- the estimate is *not yet correct*
    - its error comes from stopping early, not from randomness

We will subsequently contrast this to the iterative estimates produced by model-free methods.


**Pseudo-code for value iteration** (the algorithm of Sutton and Barto, PDF page 105, in our notation; the book's textbook version is the aside after this slide)

- initialize $\statevalfun(\state) = 0$ for every state (terminal states keep $0$ and are never updated)
- repeat
    - $\Delta \leftarrow 0$
    - for each state $\state$
        - $v_{\text{old}} \leftarrow \statevalfun(\state)$
        - $\statevalfun(\state) \leftarrow \max{\act} \sum_{\state', \rew} \transp(\state', \rew \mid \state, \act)\bigl(\rew + \disc\,\statevalfun(\state')\bigr)$
        - $\Delta \leftarrow \max{}\bigl(\Delta, \, |v_{\text{old}} - \statevalfun(\state)|\bigr)$
    - until $\Delta$ is below a small threshold
- then, for each state: $\pi(\state) \leftarrow \argmax{\act} \sum_{\state', \rew} \transp(\state', \rew \mid \state, \act)\bigl(\rew + \disc\,\statevalfun(\state')\bigr)$
- return $\pi$ and $\statevalfun$


The same algorithm, in two notations
- the pseudo-code above is in our notation
- the book's own listing (the aside after this slide) says the same thing in the book's symbols

<table>
<thead><tr><th nowrap><div align="left">meaning</div></th><th nowrap><div align="center">our notation</div></th><th nowrap><div align="center">the book</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left">value of a state</div></td><td nowrap><div align="center">$\statevalfun(\state)$</div></td><td nowrap><div align="center">$V(s)$</div></td></tr>
<tr><td nowrap><div align="left">model: chance of next state $\state'$ and reward $\rew$</div></td><td nowrap><div align="center">$\transp(\state', \rew \mid \state, \act)$</div></td><td nowrap><div align="center">$p(s', r \mid s, a)$</div></td></tr>
<tr><td nowrap><div align="left">discount</div></td><td nowrap><div align="center">$\disc$</div></td><td nowrap><div align="center">$\gamma$</div></td></tr>
<tr><td nowrap><div align="left">stopping threshold</div></td><td nowrap><div align="center">"a small threshold" on $\Delta$</div></td><td nowrap><div align="center">$\theta$</div></td></tr>
</tbody></table>
- the book says "iteration" for both a pass over the states and a whole evaluate-then-improve round
- we say **sweep** for the first and **round** for the second


**Aside: the textbook's version of value iteration** (for your own review, in the book's notation)

<br>

<table>
    <center><strong>Value iteration</strong></center>
    <tr>
        <img src="images/RL_value_iteration_alg.png" width=90%>
    </tr>
    

</table>

Attribution: http://incompleteideas.net/book/RLbook2020.pdf#page=105

## Policy iteration method

**Why a second method?** Value iteration works, but it searches over all actions in every sweep. Policy iteration organizes the same computation differently.

The Value Iteration method has one expensive step
- the $\max{\act}$ is a search over all possible actions $\act \in \Actions$

$$
\statevalfun_{k+1}(\state) = \max{\act} \sum_{\state', \rew} \transp(\state', \rew \mid \state, \act)\bigl(\rew + \disc \statevalfun_k(\state')\bigr).
$$

Policy iteration moves that cost elsewhere: it replaces the max inside every sweep by repeated full evaluations of one policy.


Policy Iteration is another way to reach the optimal $\statevalfun$ and $\pi_*$: it avoids the $\max{\act}$ inside every sweep, but pays for fully evaluating each policy.

Rather than updating the Policy once (after Value function convergence)
- we introduce a method that periodically updates the Policy.


*Policy iteration* is an algorithm that improves $\pi_p$ to $\pi_{p+1}$ by alternating two steps
during round $p$

The algorithm alternates between
- Policy evaluation
    - compute $\statevalfun_{\pi_p}$, the value of the *current* policy $\pi_p$
- Policy improvement
    - make $\pi_{p+1}$ greedy with respect to $\statevalfun_{\pi_p}$

**What changes: the max over $\act$ becomes the policy's action $\pi(\state)$**

Value iteration: the action is *searched* for, by the max
$$
\statevalfun_{k+1}(\state) = \max{\act} \sum_{\state', \rew} \transp(\state', \rew \mid \state, \act)\bigl(\rew + \disc \statevalfun_k(\state')\bigr)
$$

Policy evaluation: the action is *given*, by the current policy
$$
\statevalfun_{k+1}(\state) = \sum_{\state', \rew} \transp\bigl(\state', \rew \mid \state, \boxed{\pi(\state)}\bigr)\Bigl(\rew + \disc \statevalfun_k(\state')\Bigr)
$$

- the only change is in the boxes: the $\act$ that the max searched over is replaced by $\pi(\state)$, one fixed action per state, so there is no search inside the sweep
- the search happens once per round, in policy improvement
$$
\pi_{p+1}(\state) = \argmax{\act} \sum_{\state', \rew} \transp(\state', \rew \mid \state, \act)\bigl(\rew + \disc \statevalfun_{\pi_p}(\state')\bigr)
$$


It uses the Bellman update

$$
\statevalfun_{\pi,k+1}(\state) =  \sum_{\state', \rew} \transp(\state', \rew \mid \state, \pi(\state))\bigl(\rew + \disc \statevalfun_{\pi,k}(\state')\bigr).
$$

where
- we add an explicit subscript $\pi$
- to the value function
$$
\statevalfun_{\pi,k+1}(\state)
$$
- to indicate the dependence of the update on the *current policy* $\pi$

Rather than
- evaluating *all* actions $\act \in \Actions$
- it chooses a single action $\pi(\state)$ according to the current policy

**Policy iteration on Frozen Lake**

<img src="images/policy_iteration_rounds.png" width=100%>

- start from *always down*
- each round evaluates the current policy, then improves it greedily
- $4$ rounds reach the optimal policy
- orange arrows mark the actions that changed
- the numbers are the value of the *current* policy, so they differ from the optimal values until the last round


**Pseudo-code for policy iteration** (the algorithm of Sutton and Barto, PDF page 102, in our notation; the book's textbook version is the aside after this slide)

- initialize a policy $\pi$ arbitrarily (an arbitrary action in each state) and $\statevalfun(\state) = 0$ for every state (terminal states keep $0$)
- repeat
    - **policy evaluation** (compute $\statevalfun$ for the current $\pi$): repeat, for each state $\state$, $\statevalfun(\state) \leftarrow \sum_{\state', \rew} \transp(\state', \rew \mid \state, \pi(\state))\bigl(\rew + \disc\,\statevalfun(\state')\bigr)$, until the values change by less than a small threshold
    - **policy improvement:** policy-stable $\leftarrow$ true; for each state $\state$
        - $\act_{\text{old}} \leftarrow \pi(\state)$
        - $\pi(\state) \leftarrow \argmax{\act} \sum_{\state', \rew} \transp(\state', \rew \mid \state, \act)\bigl(\rew + \disc\,\statevalfun(\state')\bigr)$ (on a tie keep $\act_{\text{old}}$)
        - if $\act_{\text{old}} \neq \pi(\state)$, policy-stable $\leftarrow$ false
    - until policy-stable
- return $\pi$ and $\statevalfun$


**Aside: the textbook's version of policy iteration** (for your own review, in the book's notation)

<br>

<table>
    <center><strong>Policy iteration</strong></center>
    <tr>
        <img src="images/RL_policy_iteration_alg.png" width=90%>
    </tr>
    

</table>

Attribution: http://incompleteideas.net/book/RLbook2020.pdf#page=102

Both Value Iteration and Policy Iteration converge to an optimal policy (they may differ only where actions tie).

The benefit of alternating between Policy Evaluation and Policy Improvement
- fewer rounds to converge (each round costs more) 
    - the Value function under the current policy is fully evaluated
    - before the Policy is updated

On Frozen Lake (computed)
- value iteration needs $7$ sweeps (the last confirms that nothing changes)
- policy iteration from "always down" needs $4$ rounds, but $11$ evaluation sweeps in total
- fewer rounds does not mean less work


**What the counts mean**

- how the counts were made
    - sweeps are done in place, in ascending order of square
    - the values are kept between policy-iteration rounds
    - a sweep that changes nothing ends the loop
    - on a tie the policy keeps its old action (otherwise the policy could flip between equal actions forever)
- a value-iteration sweep backs up $4$ actions per state and a policy-evaluation sweep $1$, so by backups the work is similar: about $7 \times 4 = 28$ against $11 \times 1 + 4 \times 4 = 27$ per state
- round $4$ of policy iteration only confirms that nothing changes, as the last sweep of value iteration does

**Why both converge**

- the two final pictures can differ at the start square (down versus right): both routes are $6$ moves, an exact tie, so both are optimal
- why they converge: improving a policy never makes it worse, and a policy that no longer changes satisfies the optimality equation (with finitely many policies, policy iteration stops)
- caution: reaching the answer in finitely many sweeps is a feature of this small, deterministic lake with no loops
- in general the values approach the optimum only in the limit


# Interlude: evaluate versus improve

**Why this section?** Both algorithms did two things, in different orders. Naming them now gives a loop that returns in every later method.

<img src="images/gpi_loop.png" width=80%>

- **evaluate:** compute the value of a given policy (iterate the policy's Bellman equation)
- **improve:** make the policy greedy with respect to a value (the one-step lookahead of the last session)
- the two push each other
    - a better value makes a better greedy policy
    - a better policy has a different value
    - they stop when neither changes: the optimum $\statevalfun_*$ and $\pi_*$
- the two algorithms of this session are two ways of running this loop
    - **policy iteration:** evaluate *fully*, then improve, and repeat
    - **value iteration:** improve (using the current values), then do a *single* sweep of evaluation, so the two steps are merged into one update with a max
- this loop is called *generalized policy iteration*
- later methods run the same loop with samples


## Key Differences Between Value Iteration and Policy Iteration

**Why this section?** A side-by-side summary of the two algorithms we just saw, as two ways of running the loop.

Here is a comparison of the Value and Policy Iteration methods.

<table>
<thead><tr><th nowrap><div align="left">Feature</div></th><th nowrap><div align="left">Value Iteration</div></th><th nowrap><div align="left">Policy Iteration</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left">Approach</div></td><td nowrap><div align="left">Updates value function until convergence</div></td><td nowrap><div align="left">Alternates between value evaluation and improvement</div></td></tr>
<tr><td nowrap><div align="left">Convergence</div></td><td nowrap><div align="left">When value function $\statevalfun(\state)$ stabilizes</div></td><td nowrap><div align="left">When policy $\pi(\state)$ stops improving</div></td></tr>
<tr><td nowrap><div align="left">Computational Cost</div></td><td nowrap><div align="left">One sweep, with a max over actions in every state</div></td><td nowrap><div align="left">Per round: a full evaluation of the current policy (several sweeps), then one argmax over actions</div></td></tr>
<tr><td nowrap><div align="left">Speed</div></td><td nowrap><div align="left">More sweeps, each updating every state once</div></td><td nowrap><div align="left">Fewer rounds; each round costs more</div></td></tr>
<tr><td nowrap><div align="left">Policy Output</div></td><td nowrap><div align="left">Implicit in the values at every sweep; extracted at the end</div></td><td nowrap><div align="left">Updated during each round</div></td></tr>
</tbody></table>






## The limitation: dynamic programming needs the model

**Why this section?** Everything above used $\transp(\state', \rew \mid \state, \act)$. Where do they come from?

- every update **sums over all successor states, using the known model**
    - a board game's rules, or a grid we have written down, provide it
    - a robot, a market or a conversation does not
- every sweep also visits **every state**: the cost grows with the size of the state space
- so dynamic programming is exact and simple, but it needs the model and a small state space
- the next session removes the first requirement: learn the values from *samples* of experience instead


# What we established, and what it sets up

- **Established**
    - **value iteration** repeats the Bellman optimality update (with a max over actions) until the values stop changing
    - **policy iteration** alternates evaluating the current policy and improving it
    - both are loops of **evaluate** and **improve**
    - the loop returns in later methods
    - both need the known model, and sweep every state
- **Sets up**
    - next session (A3): the model is unknown, so values are learned from sampled experience (Monte Carlo and temporal difference)
    - that raises the questions of bias and variance
    - later: **control**, learning to act and not only to evaluate (A4), and a network in place of the table when it is too big (A5)
